# Midway Meeting Demo Notebook


In [1]:
import numpy as np

from matplotlib import pyplot as plt

import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient

# Locate examples

- Arrow maze choice task
- Optogenetic self-stimulation
- Open field task



In [4]:
from dandi.dandiapi import DandiAPIClient

# Get assets for a specific session
tmaze_session_id = "tmaze-2022-10-21T15-28-36_image"
dligth_session_id = "oft-2024-03-01T10-16-32"
oft_session_id = ""

def get_asset(session_id: str):
    with DandiAPIClient() as client:
        client.dandi_authenticate()
        dandiset = client.get_dandiset("001828", "draft")

        asset = next(asset for asset in dandiset.get_assets() if session_id in asset.path)
    return asset

def stream_nwb(session_id: str):
    asset = get_asset(session_id=session_id)
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)
    file_system = remfile.File(s3_url)
    file = h5py.File(file_system, mode="r")
    io = NWBHDF5IO(file=file)
    return io.read()


## Example T-maze session

`session_id = "tmaze-2022-10-21T15-28-36_image"`

### Data streams:
- Video
- Pose estimation

In [6]:
nwbfile = stream_nwb(session_id=tmaze_session_id)

print(f"Session ID:         {nwbfile.session_id}")
print(f"Session start:      {nwbfile.session_start_time}")
print(f"Session desc:       {nwbfile.session_description[:80]}...")
print(f"Subject ID:         {nwbfile.subject.subject_id}")
print(f"Species:            {nwbfile.subject.species}")
print(f"Genotype:           {nwbfile.subject.genotype}")
print(f"Sex:                {nwbfile.subject.sex}")
print(f"Institution:        {nwbfile.institution}")
print(f"Lab:                {nwbfile.lab}")
print(f"Related pubs:       {nwbfile.related_publications}")

Session ID:         tmaze-2022-10-21T15-28-36
Session start:      2022-10-21 15:28:36+02:00
Session desc:       Arrow maze choice task (T-maze) session. A water-restricted mouse navigated a T-...
Subject ID:         579834
Species:            Mus musculus
Genotype:           WT
Sex:                F
Institution:        Karolinska Institutet
Lab:                Meletis
Related pubs:       ('doi:10.1101/2024.12.22.629963',)


## 1. Video (external reference)

Videos are stored as external file references rather than embedded pixel data.
Each session has one MP4 video (H.264, 30 fps, ~856×818 pixels) recorded from below
through a transparent floor. The `external_file` field contains the relative path to the video.

In [7]:
video_name = "Video tmaze_2022-10-21T15_28_36"

video = nwbfile.acquisition[video_name]
print(f"Video name:      {video.name}")
print(f"External file:   {video.external_file[0]}")
print(f"Starting time:   {video.starting_time} s")
print(f"Rate:            {video.rate} fps")
print(f"Description:     {video.description}")

Video name:      Video tmaze_2022-10-21T15_28_36
External file:   sub-579834_ses-tmaze-2022-10-21T15-28-36_image/d9c4acf2-4834-41cf-bd5e-b1fbce3d78db_external_file_0.mp4
Starting time:   0.0 s
Rate:            30.0 fps
Description:     Video recorded by camera.


In [9]:
from nwb_video_widgets import NWBDANDIVideoPlayer

asset = get_asset(tmaze_session_id)

NWBDANDIVideoPlayer(asset=asset)

## 2. Pose estimation (DeepLabCut)

Pose estimation was performed using DeepLabCut, tracking 8 keypoints on each mouse:
**snout**, **head**, **body**, **left front paw**, **right front paw**, **left back paw**,
**right back paw**, and **tail base**. Stored via the
[ndx-pose](https://github.com/rly/ndx-pose) extension.

Each keypoint has:
- `data` — (N, 2) array of (x, y) pixel coordinates at 30 Hz
- `confidence` — (N,) array of DLC likelihood scores per frame

Scorer: `DLC_resnet50_completeArrowMazeDec9shuffle1_1000000`

In [ ]:
behavior = nwbfile.processing["behavior"]
pose = behavior["PoseEstimationDeepLabCut"]

keypoint_names = list(pose.pose_estimation_series.keys())
print(f"Keypoints ({len(keypoint_names)}):")
for kp in keypoint_names:
    pes = pose.pose_estimation_series[kp]
    conf = pes.confidence[:]
    print(f"  {kp.replace('PoseEstimationSeries', ''):20s}  shape={str(pes.data.shape):12s}  "
          f"conf_mean={conf.mean():.3f}")

print(f"\nScorer:          {pose.scorer}")
print(f"Source software: {pose.source_software}")

In [ ]:
# Plot body-center trajectory colored by time
body_xy = pose.pose_estimation_series["PoseEstimationSeriesBody"].data[:]
head_xy = pose.pose_estimation_series["PoseEstimationSeriesHead"].data[:]

body_xy = body.data[:]
n_frames = len(body_xy)
fps = body.rate  # 30.0 Hz
time_s = np.arange(n_frames) / fps

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trajectory colored by time
sc = axes[0].scatter(body_xy[:,0], body_xy[:, 1], c=time_s, s=0.5, cmap="viridis", alpha=0.5)
axes[0].set_xlabel("x (pixels)")
axes[0].set_ylabel("y (pixels)")
axes[0].set_title("Body-center trajectory")
axes[0].set_aspect("equal")
plt.colorbar(sc, ax=axes[1], label="Time (s)")
axes[0].invert_yaxis()
axes[0].axis("off")

axes[1].scatter(head_xy[:,0], head_xy[:, 1], c=time_s, s=0.5, cmap="viridis", alpha=0.5)
axes[1].set_title("Head trajectory")
axes[1].invert_yaxis()
axes[1].axis("off")
plt.show()

In [11]:
from nwb_video_widgets import NWBDANDIPoseEstimationWidget

asset = get_asset(tmaze_session_id)

NWBDANDIPoseEstimationWidget(asset=asset)

## Example fiber photometry session

`session_id = "oft-2024-03-01T10-16-32"`


### Data streams:
- Fiber photometry
- Optogenetics

In [13]:
nwbfile = stream_nwb(session_id=dligth_session_id)

print(f"Session ID:         {nwbfile.session_id}")
print(f"Session start:      {nwbfile.session_start_time}")
print(f"Session desc:       {nwbfile.session_description[:80]}...")
print(f"Subject ID:         {nwbfile.subject.subject_id}")
print(f"Species:            {nwbfile.subject.species}")
print(f"Genotype:           {nwbfile.subject.genotype}")
print(f"Sex:                {nwbfile.subject.sex}")
print(f"Institution:        {nwbfile.institution}")
print(f"Lab:                {nwbfile.lab}")
print(f"Related pubs:       {nwbfile.related_publications}")

Session ID:         oft-2024-03-01T10-16-32
Session start:      2024-03-01 10:16:32+01:00
Session desc:       Optogenetic self-stimulation session. A water-restricted mouse freely nosepoked ...
Subject ID:         776769
Species:            Mus musculus
Genotype:           Anxa1-flp
Sex:                F
Institution:        Karolinska Institutet
Lab:                Meletis
Related pubs:       ('doi:10.1101/2024.12.22.629963',)


In [15]:
nwbfile

Data type,float64
Shape,"(74763,)"
Array size,584.09 KiB
Chunk shape,"(74763,)"
Compression,gzip
Compression opts,4
Uncompressed size (bytes),598104
Compressed size (bytes),228553
Compression ratio,2.616915988851601
Data type,float64
Shape,"(74763,)"
